# Fusion Categories: How Anyons Combine -- Stakeholder Edition

**Phase 5c, Wave 3** | SK-EFT Hawking Project -- Stakeholder Edition

## What This Is About

When you combine two particles in ordinary physics, the rules are simple: two spin-1/2
particles can form spin-0 or spin-1. But in two-dimensional systems, particles called
**anyons** follow different rules -- and these rules are described by **fusion categories**.

This notebook explains what fusion categories are, why the Ising and Fibonacci cases
matter for quantum computing, and how the Verlinde formula provides an independent
cross-check on the entire structure.

In [ ]:
import numpy as np
from src.core.formulas import (
    su2k_fusion_rule, su2k_quantum_dim, su2k_global_dim_sq,
    su2k_s_matrix_entry, su2k_verlinde
)
from src.core.visualizations import fig_su2k_fusion_tables, fig_su2k_quantum_dims, COLORS

## What Is a Fusion Category?

A **fusion category** is a set of particle types together with rules for what happens
when you bring two particles together. In math notation:

$$a \times b = \sum_c N_{ab}^c \, c$$

The numbers $N_{ab}^c$ tell you how many ways particles $a$ and $b$ can fuse to produce
particle $c$. For the simplest theories (like SU(2)$_k$), these numbers are 0 or 1.

**Key difference from ordinary particles:** In 3D, combining particles always gives a
unique result (determined by conservation laws). In 2D, combining anyons can give
**multiple possible outcomes** -- and this multiplicity is what enables quantum computation.

Think of it like chemistry: combining hydrogen and oxygen can give water (H$_2$O) or
hydrogen peroxide (H$_2$O$_2$). Fusion rules tell you what "molecules" are possible.

In [ ]:
# Show the three simplest fusion theories
theories = {
    1: ('Semion', 'Simplest non-trivial theory'),
    2: ('Ising', 'Majorana zero modes'),
    3: ('Fibonacci', 'Universal quantum computing'),
}

for k, (name, significance) in theories.items():
    n = k + 1
    print(f'--- SU(2)_{k}: {name} ({n} particle types) ---')
    print(f'    Significance: {significance}')
    for i in range(n):
        for j in range(i, n):
            channels = [f'V_{m}' for m in range(n) if su2k_fusion_rule(k, i, j, m) == 1]
            if channels:
                decomp = ' + '.join(channels)
                print(f'    V_{i} x V_{j} = {decomp}')
    print()

In [ ]:
# viz-ref: fig_su2k_fusion_tables
fig = fig_su2k_fusion_tables()
fig.show()

## Why Ising Anyons Matter: Majorana Zero Modes

The $k = 2$ theory (Ising) has three particle types:
- $V_0 = 1$ (vacuum, does nothing)
- $V_1 = \sigma$ (the "interesting" anyon, quantum dimension $\sqrt{2}$)
- $V_2 = \psi$ (a fermion)

The key fusion rule is: $\sigma \times \sigma = 1 + \psi$

Two Ising anyons can fuse to vacuum OR to a fermion. This 2-fold ambiguity stores
one bit of quantum information that is **topologically protected** -- no local
perturbation can flip it. This is the basis of Microsoft's approach to quantum computing
using Majorana zero modes in nanowires.

The quantum dimension $d_\sigma = \sqrt{2}$ tells you: each Ising anyon contributes
$\sqrt{2}$ to the Hilbert space dimension. Two anyons give $\sqrt{2}^2 = 2$ states,
which is the qubit.

In [ ]:
# Quantum dimensions: the "size" of each anyon
print('Quantum dimensions (the "size" of each particle type):')
print('=' * 55)
for k in [1, 2, 3]:
    n = k + 1
    names = {1: 'Semion', 2: 'Ising', 3: 'Fibonacci'}
    dims = [su2k_quantum_dim(k, j) for j in range(n)]
    D2 = su2k_global_dim_sq(k)
    dim_str = ', '.join(f'd_{j}={d:.3f}' for j, d in enumerate(dims))
    print(f'  k={k} ({names[k]:>9}): {dim_str}')
    print(f'    Total quantum dimension D^2 = {D2:.4f}')

print(f'\nSpecial values:')
print(f'  Ising sigma:     d = {su2k_quantum_dim(2, 1):.4f} = sqrt(2)')
print(f'  Fibonacci tau:   d = {su2k_quantum_dim(3, 1):.4f} = golden ratio')

In [ ]:
# viz-ref: fig_su2k_quantum_dims
fig = fig_su2k_quantum_dims()
fig.show()

## Why Fibonacci Anyons Matter: Universal Quantum Computing

The $k = 3$ theory contains the **Fibonacci anyon** $\tau$ with the fusion rule:

$$\tau \times \tau = 1 + \tau$$

This is the simplest fusion rule that enables **universal** topological quantum computation.
"Universal" means: by braiding Fibonacci anyons, you can approximate *any* quantum gate
to arbitrary precision. No other ingredient is needed.

The golden ratio $\varphi = (1+\sqrt{5})/2 \approx 1.618$ appears as the quantum dimension
because the fusion rule $\tau^2 = 1 + \tau$ is the defining equation of the golden ratio:
$\varphi^2 = 1 + \varphi$.

**Connection to our project:** SU(2)$_3$ arises from the quantum group $U_q(\mathfrak{sl}_2)$
at $q = e^{i\pi/5}$ (5th root of unity). The Hopf algebra structure from Wave 1 is what
produces this fusion category at the root of unity specialization.

## The Verlinde Formula: An Independent Cross-Check

There are two completely independent ways to compute fusion rules:

1. **Representation theory:** Use truncated Clebsch-Gordan rules (like adding angular momenta
   with a cutoff)
2. **Modular S-matrix:** Use the Verlinde formula $N_{ij}^m = \sum_l S_{il} S_{jl} S_{ml} / S_{0l}$

The fact that both methods give the same answer is a powerful consistency check. It means
the fusion rules are not arbitrary -- they are forced by the underlying symmetry structure.

**Analogy:** Imagine computing the area of a field by (1) walking the perimeter and using
surveying formulas, and (2) taking a satellite photo and counting pixels. If both methods
agree, you can be confident the answer is correct.

In [ ]:
# Verlinde cross-validation: two independent paths to the same answer
print('Verlinde cross-validation: do two independent methods agree?')
print('=' * 60)

for k in [1, 2, 3]:
    n = k + 1
    names = {1: 'Semion', 2: 'Ising', 3: 'Fibonacci'}
    all_match = True
    count = 0
    for i in range(n):
        for j in range(n):
            for m in range(n):
                n_cg = su2k_fusion_rule(k, i, j, m)
                n_verl = round(su2k_verlinde(k, i, j, m))
                count += 1
                if n_cg != n_verl:
                    all_match = False
    status = 'PERFECT MATCH' if all_match else 'MISMATCH'
    print(f'  k={k} ({names[k]:>9}): {count} coefficients checked => {status}')

print(f'\nBoth methods agree perfectly -- the fusion rules are consistent.')